# MiloR differential abundance testing

In [4]:
# Load packages
suppressPackageStartupMessages({
    library(dplyr)
    library(data.table)
    library(ggplot2)
    library(SingleCellExperiment)
    library(scater)
    library(scran)
    library(edgeR)
    library(ggrastr)
    library(Seurat)
})

here::i_am("dimensionality_reduction/01_dimensionality_reduction_manual.ipynb")

# Load default settings
source(here::here("settings.R"))
source(here::here("utils.R"))
BPPARAM <- BiocParallel::bpparam()
BPPARAM$workers = 21

set.seed(1234)

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/14_Yang/code



In [5]:
io$basedir

[1] "/rds/project/rds-SDzz0CATGms/users/bt392/14_Yang"

In [6]:
args = list()
args$seurat_yang = '/rds/project/rds-SDzz0CATGms/users/ltgh2/projects/17_Yang_Histone_Atlas/seurat_object_yang/scRNA.data.seurat.all.202411.processed2.rds'

args$batch_correction = "sample"
args$vars_to_regress = c('nFeature_RNA', 'nCount_RNA', 'percent.mt')
args$features = 2500
args$npcs = 40
args$n_neighbors = 20
args$min_dist = 0.4
args$prop = 0.1
args$seed = 12345
args$outdir = paste0(io$basedir,"/results/rna/dimensionality_reduction/manual/")
dir.create(args$outdir, recursive=TRUE, showWarnings = FALSE)

# Dimensionality reduction

In [7]:
################
## Load query ##
################

seurat = readRDS(args$seurat_yang)

count.mtx = seurat@assays$RNA@layers$counts
colnames(count.mtx) = rownames(seurat@assays$RNA@cells)
rownames(count.mtx) = rownames(seurat@assays$RNA@features)

sce_query = SingleCellExperiment(list(counts = count.mtx))
colData(sce_query) <- seurat@meta.data %>% DataFrame

metadata = as.data.table(colData(sce_query), keep.rownames = T) %>% setnames('rn', 'cell')

sce_query = sce_query[, metadata[nCount_RNA > 500 & nCount_RNA < 20000 & nFeature_RNA > 50, cell]]

metadata = metadata[match(colnames(sce_query), cell)]

sce_query$sample = paste0(metadata$orig.ident, '_', 
          #  metadata$sample_bio.id, '_', 
            metadata$sublib, '_', 
            metadata$batches
           )

meta_query = metadata
meta_query$sample = sce_query$sample

Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”


In [8]:
sample_metadata = meta_query
sce = sce_query

In [9]:
sce = logNormCounts(sce)

In [10]:
##############################
## Dimensionality Reduction ##
##############################

if (args$batch_correction %in% c('sample')) {
     source(here::here("mapping/run/mnn/mapping_functions.R"))
     library(batchelor)
   }

 if (length(args$vars_to_regress)>0) {
  stopifnot(args$vars_to_regress%in%colnames(sample_metadata))
 }

## Feature selection 

# Find HVGs - detection on WT samples only
# Get gene metadata
gene_metadata <- fread(io$gene_metadata) %>% .[,c("chr","ens_id","symbol", "description")] %>%
  .[symbol!="" & ens_id%in%rownames(sce)] %>%
  .[!duplicated(symbol)]

# Imprinted genes
imprint = gene_metadata[c(grep('maternally', gene_metadata$description),
                       grep('paternally', gene_metadata$description)), symbol]
# Other imprinted genes: 
#- Nnat (https://www.genecards.org/cgi-bin/carddisp.pl?gene=NNAT)
#- Grb10 (https://www.genecards.org/cgi-bin/carddisp.pl?gene=GRB10)

genes_keep = rownames(sce)
# genes_keep <- genes_keep[grep("^Rik|Rik$|^mt-|^Rps|^Rpl|^Gm",genes_keep,invert=T)] # filter out non-informative genes
# genes_keep <- genes_keep[grep("^Hbb|^Hba",genes_keep,invert=T)] # test removing Haem genes 
# genes_keep <- genes_keep[!genes_keep %in% c(imprint, 'Grb10', 'Nnat')] # remove imprinted genes
genes_keep <- genes_keep[!genes_keep %in% c("Xist", "Tsix")] # remove Xist & Tsix
# genes_keep <- genes_keep[!genes_keep == "tomato-td"] # remove tomato itself
genes_keep <- genes_keep[!genes_keep %in% gene_metadata[chr=="chrY",symbol]] # no genes on y-chr 

# Find variable genes using Seurat
# hvgs = VariableFeatures(FindVariableFeatures(as.Seurat(sce), nfeatures = args$features))

# sce_filt <- sce[hvgs,]

In [40]:
decomp <- modelGeneVar(sce, block = sce$sample)
decomp <- decomp[decomp$mean > 0.01,]
hvgs <- decomp[order(decomp$FDR, -decomp$bio),] %>% head(n=args$features) %>% rownames

sce_filt <- sce[hvgs,]

In [41]:
args$vars_to_regress = NULL

In [42]:
## Regress out covariates 
 if (length(args$vars_to_regress)>0) {
   print(sprintf("Regressing out variables: %s", paste(args$vars_to_regress,collapse=" ")))
   logcounts(sce_filt) <- RegressOutMatrix(
     mtx = logcounts(sce_filt), 
     covariates = colData(sce_filt)[,args$vars_to_regress,drop=F], ncores = 1 # parallelization wrong so single-core faster than multi
   )
 saveRDS(logcounts(sce_filt), file.path(args$outdir, 'logcounts_regressed.rds'))
 }

In [45]:
print('batch correcting by sample')
pca <- multiBatchPCA(sce_filt, batch = colData(sce_filt)[[args$batch_correction]], d = args$npcs)

[1] "batch correcting by sample"


In [ ]:
pca.corrected <- reducedMNN(pca, auto.merge = T)$corrected
colnames(pca.corrected) <- paste0("PC",1:ncol(pca.corrected))
pca.corrected = pca.corrected[match(colnames(sce_filt), rownames(pca.corrected)),]
reducedDim(sce_filt, "PCA") <- pca.corrected

In [44]:
head(colData(sce_filt)[[args$batch_correction]])

[1] "S2_S2_batch3" "S2_S2_batch3" "S2_S2_batch3" "S2_S2_batch3" "S2_S2_batch3"
[6] "S2_S2_batch3"

In [ ]:
a

In [ ]:
pca.dt <- reducedDim(sce_filt,"PCA") %>% as.data.table(keep.rownames = T) %>% setnames("rn","cell")


In [ ]:
fwrite(pca.dt, file.path(args$outdir, 'PCA.txt.gz'))

In [ ]:
## UMAP
set.seed(args$seed)
sce_filt <- runUMAP(sce_filt, dimred="PCA", n_neighbors = args$n_neighbors, min_dist = args$min_dist)

# Fetch UMAP coordinates
umap.dt <- reducedDim(sce_filt,"UMAP") %>% as.data.table %>% 
  .[,cell:=colnames(sce_filt)] %>%
  setnames(c("UMAP1","UMAP2","cell"))

In [ ]:
fwrite(umap.dt, file.path(args$outdir, 'UMAP.txt.gz'))

In [ ]:
# plotting
to.plot <- reducedDim(sce_filt,"UMAP") %>% as.data.table %>% 
    .[,cell:=colnames(sce_filt)] %>%
    merge(sample_metadata, by="cell")

In [ ]:
ggplot(to.plot, aes_string(x="V1", y="V2")) +
    geom_point(size=0.2) +
    theme_classic() +
    # scale_color_manual(values=c(opts$celltype.colors, opts$celltype_extended.colors)) + 
    theme(
        legend.position="none",
        legend.title=element_blank(),
        axis.title = element_blank(),
        axis.text = element_blank(),
        axis.ticks = element_blank()
    )

ggplot(to.plot, aes_string(x="V1", y="V2", col='sample')) +
    geom_point(size=0.2) +
    theme_classic() +
    # scale_color_manual(values=opts$tdTom.color) +
    theme(
        legend.position="none",
        legend.title=element_blank(),
        axis.title = element_blank(),
        axis.text = element_blank(),
        axis.ticks = element_blank()
    )